# 11 — Export to ONNX and run without PyTorch

The capstone: turn a Hub checkpoint into an ONNX graph, run it with ONNX Runtime, and check that it agrees with the PyTorch model. The exported folder carries its own pre- and post-processing, so the consumer needs only numpy, Pillow, and ONNX Runtime — no PyTorch, no OrientedDet.

| | |
|---|---|
| Needs | Internet; the `[export]` extras (`onnx`, `onnxruntime`), installed by the setup cell |
| Kaggle | Yes — Internet ON |
| Scope | **Fixed 1024×1024 canvas** (DOTA checkpoints). RetinaNet and `keep_ratio` checkpoints (HRSC, SSDD, HRSID) are not exported yet. |
| Docs | [ONNX export guide](https://github.com/DL4EO/oriented-det/blob/main/docs/examples/export.md) · [export/README.md](https://github.com/DL4EO/oriented-det/blob/main/export/README.md) · [export/PARITY.md](https://github.com/DL4EO/oriented-det/blob/main/export/PARITY.md) |
| Post | [v0.3.0: four datasets and ONNX](https://deeplearning.earth/posts/2026-09-21_oriented-det_v0_3_0_four_datasets_and_onnx/) |

## What gets exported

The ONNX graph contains the network up to its **pre-NMS candidates** on a fixed 1024×1024 input: backbone, neck, head, and box decoding. Everything that depends on image size or needs rotated geometry stays outside the graph, in small numpy modules copied next to `model.onnx`:

1. `preprocess.py` resizes and normalises the image exactly as in training (values stored in `model.meta.json`);
2. ONNX Runtime runs the graph;
3. `postprocess.py` + `nms.py` apply the score floor, rotated NMS, and map boxes back to original pixels.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det[export]>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}[export]"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

In [ ]:
import importlib.util

if importlib.util.find_spec("onnxruntime") is None or importlib.util.find_spec("onnx") is None:
    %pip install -q "onnx>=1.14,<2" "onnxruntime>=1.16"

## 1. Export a Hub checkpoint

Rotated FCOS is the default: a single-stage graph with no proposal sampling, the simplest to export.

In [ ]:
SLUG = "rotated_fcos_dota_le90_1x"
ONNX_DIR = WORK / "11_onnx"
onnx_path = ONNX_DIR / "model.onnx"

ckpt, sidecar = hub_files(SLUG)
run([sys.executable, "-m", "export", "onnx", "--config", sidecar, "--checkpoint", ckpt, "--output", onnx_path])
for p in sorted(ONNX_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(ONNX_DIR)}")

## 2. Inspect the graph and its metadata

In [ ]:
import json
import onnx
from export.runtime import load_export_meta

graph = onnx.load(str(onnx_path)).graph
for kind, values in (("input", graph.input), ("output", graph.output)):
    for v in values:
        dims = [d.dim_value or d.dim_param for d in v.type.tensor_type.shape.dim]
        print(f"{kind:6s} {v.name:24s} {dims}")
meta = load_export_meta(onnx_path)
print(json.dumps({k: v for k, v in meta.items() if k != "class_names"}, indent=1)[:1500])

## 3. Detect with ONNX Runtime

In [ ]:
from PIL import Image
from export.runtime import detect_image, draw_detections

image = Image.open(demo_file("demo.jpg")).convert("RGB")
detect_image(image, onnx_path, meta)  # warm-up (session creation)
t0 = time.perf_counter()
onnx_dets = detect_image(image, onnx_path, meta)
print(f"{len(onnx_dets)} detections in {time.perf_counter() - t0:.2f} s (ONNX Runtime)")
print(onnx_dets[0])
show(draw_detections(image, onnx_dets), titles=(f"ONNX Runtime: {len(onnx_dets)} detections",))

## 4. Parity with PyTorch

Run the same checkpoint in PyTorch with `detect()` and match every ONNX box to its best PyTorch box. Small differences are expected (float32 kernels, resize implementations); `export/PARITY.md` documents the measured mAP difference on DOTA val.

In [ ]:
import numpy as np
from oriented_det.geometry import RBox
from oriented_det.ops import rbox_iou

model, config, class_names = hub_model(SLUG)
torch_dets = detect(model, config, class_names, image)
ious = np.array([max((rbox_iou(RBox(d["cx"], d["cy"], d["w"], d["h"], d["angle_rad"]), t["rbox"]) for t in torch_dets),
                     default=0.0) for d in onnx_dets])
print(f"PyTorch {len(torch_dets)} vs ONNX {len(onnx_dets)} detections")
print(f"ONNX boxes with a PyTorch match at IoU >= 0.9: {(ious >= 0.9).mean():.1%}  (median IoU {np.median(ious):.3f})")

## 5. Ship it

`ONNX_DIR` is self-contained: copy it to a machine with `pip install -r requirements-runtime.txt` (numpy, Pillow, onnxruntime) and run `python infer_onnx.py` or `python demo.py` there. Inside a checkout, `python -m export infer --onnx model.onnx --images <folder> --output <folder>` does the same for a folder of tiles.

## What to try next

- Export `oriented_rcnn_dota_le90_1x` (two-stage) and compare latency and parity with FCOS.
- Feed a larger image: the runtime resizes to the 1024 canvas, so tile it first with the window rule from lesson 06.
- Read [export/PARITY.md](https://github.com/DL4EO/oriented-det/blob/main/export/PARITY.md) for the full DOTA val comparison.
- **Back to the start:** [tutorial/README.md](README.md) lists every lesson and what it covers.